# Lab 11 · Tune versus retrieve, settled

**Day 4 · S19** · Budget: 25 min of the 40 min slot · Runs on: Colab or a laptop. CPU is enough unless you serve Day 2's adapter yourself

Since Monday the room has been holding two answers to the same question.

On Day 2 you changed the model's weights, and ticket records started coming out in the right shape every time. On Day 3 you left the weights alone, put the plant's documents in front of the model, and it started quoting setpoints correctly. Both worked. Both were sold as *the* way to make a general model do your job.

This lab runs them against each other on one task set. The argument does not survive the table.

### The task

The service desk's own: a free-text ticket in, one record out. Six fields, and they split in two.

| Field | Settled by |
|---|---|
| `ticket_id`, `category`, `affected_system` | the ticket text and the desk's own conventions |
| `priority`, `action`, `reference` | the plant's documents, where a document covers it |

That split is the whole lab. The first half is a **behaviour** problem. The second is a **knowledge** problem. They have different fixes, and neither fix touches the other half.

### Four arms, two switches

|  | no documents | with retrieval |
|---|---|---|
| **base model** | what you had on Sunday | Day 3 |
| **tuned model** | Day 2 | the one nobody argues about |

Tuning and retrieval are not two answers to one question. They are two switches, and this notebook flips both.

### What this lab loads from the other labs

| From | What | If you do not have it |
|---|---|---|
| Lab 07 | `artifacts/rag_index`, the 74-document index | run 07 to the end, or pull the folder from the repo |
| Day 2 | the tuned adapter | the notebook falls back twice, and prints which fallback it used |

## 1. Setup

Three cells: find the lab folder, pick the base model, load this morning's index.

In [2]:
# Setup: find the lab folder, detect the runtime, install pinned packages on Colab.
import os
import re
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = os.environ.get("LAB_REPO_URL", "")  # Colab: the course repo URL, once it is published


def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "scripts" / "rag_index.py").exists():
            return p


ROOT = find_root()
if ROOT is None and IN_COLAB and REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, "/content/lab"], check=True)
    ROOT = Path("/content/lab")
if ROOT is None:
    raise RuntimeError("Lab folder not found. Open this notebook from inside it, or set LAB_REPO_URL on Colab.")
if IN_COLAB:  # Colab already ships torch and sentence-transformers; rank-bm25 is the lexical half of the index
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "openai==3.0.0", "python-dotenv==1.1.0", "rank-bm25==0.2.2"], check=True)
sys.path.insert(0, str(ROOT / "scripts"))
print("lab folder:", ROOT, "| runtime:", "Colab" if IN_COLAB else "local")

lab folder: /Users/drpreetyrai./aiguru | runtime: local


In [3]:
import json

import numpy as np
import pandas as pd
from vision_client import (ask, ensure_ollama, load_openai_key, ollama_up, prebaked_models,
                           promote_to_prebaked, run_batch, self_hosted, vendor_api)

pd.set_option("display.max_colwidth", 80)

RUN_MODE = os.environ.get("LAB_RUN_MODE", "live")  # "live" calls the models, "prebaked" replays a saved run
LAB = "11"
OUT = ROOT / "outputs"                 # one folder per arm: outputs/11_base, 11_tuned, 11_rag, 11_tuned_rag
PREBAKED = Path(os.environ.get("LAB_PREBAKED_DIR", ROOT / "facilitator" / "prebaked_outputs"))
BASE_OLLAMA = os.environ.get("LAB_BASE_MODEL", "qwen2.5:3b")  # the untuned sibling of Day 2's model

BASE = None
if RUN_MODE == "live":
    if os.environ.get("LAB_BASE_MODEL") or ollama_up():  # Day 2 left Ollama running: use the same model family
        try:
            ensure_ollama(BASE_OLLAMA)
            BASE = self_hosted(BASE_OLLAMA)
        except Exception as e:
            print("self-hosted model unavailable:", e)
    if BASE is None and load_openai_key(ROOT):
        BASE = vendor_api()
if BASE is None:
    baked = prebaked_models(PREBAKED / f"{LAB}_base")
    BASE = baked[0] if baked else None
assert BASE is not None, f"No live model and no prebaked run in {PREBAKED / (LAB + '_base')}. Ask the facilitator."
WORKERS = 4 if BASE.backend == "openai" else 1
print("base model:", BASE.label)

base model: vendor API: gpt-4.1-mini


In [4]:
from rag_index import RagIndex

TEXT = RagIndex.load(ROOT / "artifacts" / "rag_index")
print(f"index: {len(TEXT.chunks)} chunks from {TEXT.manifest['documents']} documents"
      f" | embeddings {TEXT.manifest['embed_model']} | rerank {TEXT.manifest['rerank_model']}")
print("The embedder and the reranker load on the first search, which takes about a minute.")

index: 342 chunks from 74 documents | embeddings BAAI/bge-small-en-v1.5 | rerank cross-encoder/ms-marco-MiniLM-L-12-v2
The embedder and the reranker load on the first search, which takes about a minute.


## 2. The record

One schema, six fields, two kinds of field. `FROM_TICKET` is everything a reader could fill in with the ticket in front of them and no other document open. `FROM_DOCS` is everything that needs the plant's paperwork.

A schema on its own does not say what a category means or when something is priority 1. That is the desk's house style, and it is written out below because every arm is given the same rules. The only thing that varies between arms is whether the **documents** are in the prompt.

In [5]:
CATEGORIES = ["access", "network", "historian", "workstation", "control_system",
              "field_instrument", "plant_equipment", "procedure", "other"]

SCHEMA = {
    "type": "object",
    "properties": {
        "ticket_id": {"type": "string"},
        "category": {"type": "string", "enum": CATEGORIES},
        "affected_system": {"type": "string"},
        "priority": {"type": "integer"},
        "action": {"type": "string"},
        "reference": {"type": "string"},
    },
    "required": ["ticket_id", "category", "affected_system", "priority", "action", "reference"],
    "additionalProperties": False,
}
FROM_TICKET = ["ticket_id", "category", "affected_system"]  # behaviour: the desk's conventions
FROM_DOCS = ["priority", "action", "reference"]             # knowledge: the plant's documents

DESK_RULES = """\
Category, one of:
  access           user accounts, passwords, permissions
  network          switches, firewall rules, remote access paths
  historian        HS-01, HS-02, interface nodes, tags and archives
  workstation      DCS operator and engineering workstations
  control_system   DCS, SIS, the fire and gas panel
  field_instrument detectors, transmitters and valves in the field
  plant_equipment  pumps, compressors, generators and other plant
  procedure        a question about a written procedure or permit
  other            anything else, including office IT

Priority:
  1  a safety function or the plant's ability to produce is lost or bypassed right now
  2  a safety or production system is degraded, or a workaround is holding it up
  3  a question, or a fault with a workaround already in place
  4  a request for equipment, access or information, with no fault
  A priority stated in a plant document wins over these rules.

Reference: the doc_id of the document that settles the action (MAN-..., HSE-PRO-..., WO-...,
  PLAN-..., RCA-...), or NONE if no plant document covers it.
"""
# TODO(contract 4): when Day 2's schema lands in data/finetune/schema.json, load it here instead of
# declaring it. The two must not drift, or the tuned arm is being scored against a schema it never saw.
print(f"{len(SCHEMA['required'])} fields: {', '.join(FROM_TICKET)} | {', '.join(FROM_DOCS)}")

6 fields: ticket_id, category, affected_system | priority, action, reference


## 3. The ticket set

Twelve tickets, one morning's queue. They are grouped by what the documents contribute, because that is what decides which arm can win.

| Group | n | What it tests |
|---|---|---|
| `desk` | 2 | routine work no plant document covers. The correct `reference` is NONE |
| `coded` | 3 | the ticket quotes an error code whose meaning, priority and action are in a manual |
| `stale` | 2 | the governing procedure was revised. The old value is still the plausible one |
| `absent` | 2 | sounds answerable, is not. A neighbouring fact is in the corpus and is not the one asked for |
| `mixed` | 3 | messy text, two systems, a code typed from a photo, and a document behind it |

`desk` and `absent` are there to keep retrieval honest. A retriever always returns something: four passages come back for a jammed printer exactly as they do for a gas alarm, and a model that cites one of them has invented a reference. That failure is invisible in a score that only counts what retrieval finds.

In [6]:
# @title Materialise the S19 ticket set (skip-safe: never overwrites a committed file) { display-mode: "form" }
# data/eval/service_tickets.jsonl : the ticket, the record it should produce, and how each field is scored.
# Text fields are scored by regex, not string equality: there are several right ways to name a thing.
TICKETS_PATH = ROOT / "data" / "eval" / "service_tickets.jsonl"
NONE_RE = r"^\s*(none|n/?a|-|null|not applicable|no document)\s*$"
RESTART = r"(^|[.;]\s*|,\s*|and\s+|then\s+|please\s+)(restart|reboot|bounce)\b"  # not "do not restart"

TICKETS = [
    # --- desk: the plant document set says nothing. The right reference is NONE ---------------------
    dict(id="T01", kind="desk",
         text="Ticket SD-2026-0401 | raised 2026-09-28 08:12 | from: Maintenance planning\n"
              "The printer in the maintenance office jams on every second page. We are printing job packs "
              "on the planner's printer in the meantime.",
         gold=dict(ticket_id="SD-2026-0401", category="other", affected_system=r"print",
                   priority=3, action_must=[r"print|toner|jam|roller"], action_must_not=[],
                   reference=NONE_RE)),

    dict(id="T02", kind="desk",
         text="Ticket SD-2026-0402 | raised 2026-09-28 08:40 | from: Process engineering\n"
              "New graduate engineer starts on Sunday. Please provide a second monitor and a docking station "
              "for desk 14 in the engineering office. No rush.",
         gold=dict(ticket_id="SD-2026-0402", category="other", affected_system=r"monitor|dock|desk",
                   priority=4, action_must=[r"monitor|dock"], action_must_not=[], reference=NONE_RE)),

    # --- coded: the manual states the meaning, the priority and the action -------------------------
    dict(id="T03", kind="coded",
         text="Ticket SD-2026-0405 | raised 2026-09-28 09:05 | from: Control room, shift B\n"
              "The fire and gas panel has been showing FGP-E12 since about 04:00. Night shift acknowledged "
              "and reset it twice and it keeps coming back. What do we do with it?",
         gold=dict(ticket_id="SD-2026-0405", category="control_system",
                   affected_system=r"fgp|fire and gas|f&g|panel",
                   priority=1,  # MAN-FGP-01: loop 2 carries the compressor house H2S detectors
                   action_must=[r"portable gas|gas monitoring|supervisor|instrument technician"],
                   action_must_not=[], reference=r"MAN-FGP-01")),

    dict(id="T04", kind="coded",
         text="Ticket SD-2026-0409 | raised 2026-09-28 09:20 | from: OT support\n"
              "HS-01 has been logging HX-4471 since Saturday and trends are running about two hours behind. "
              "Can we just restart the historian service to clear it?",
         gold=dict(ticket_id="SD-2026-0409", category="historian", affected_system=r"hs-?0?1|historian",
                   priority=2,  # MAN-HIS-01 states priority 2, and states not to restart
                   action_must=[r"free space|archive volume|disk|storage"],
                   action_must_not=[RESTART], reference=r"MAN-HIS-01")),

    dict(id="T05", kind="coded",
         text="Ticket SD-2026-0412 | raised 2026-09-28 10:02 | from: Instrument technician\n"
              "Gas detector GD-3107 failed its six-monthly calibration this morning, span reading about 30 % "
              "low. It is inhibited at the panel and we have a portable monitor at the location. What has to "
              "happen before it goes back in service?",
         gold=dict(ticket_id="SD-2026-0412", category="field_instrument", affected_system=r"gd-?3107",
                   priority=2, action_must=[r"sensor"], action_must_not=[], reference=r"MAN-GD-01")),

    # --- stale: the procedure was revised, and the superseded value is the plausible one ------------
    dict(id="T06", kind="stale",
         text="Ticket SD-2026-0415 | raised 2026-09-28 10:30 | from: Contractor supervisor, Unit 200\n"
              "Our own H2S monitors arrived today. What do we set the low alarm to, and above what "
              "concentration do your rules require SCBA?",
         gold=dict(ticket_id="SD-2026-0415", category="procedure", affected_system=r"h2s|monitor|scba",
                   priority=3, action_must=[r"\b5\s*ppm", r"\b15\s*ppm"],
                   action_must_not=[r"\b10\s*ppm", r"\b20\s*ppm"],  # HSE-PRO-007 rev 3, superseded
                   reference=r"HSE-PRO-007")),

    dict(id="T07", kind="stale",
         text="Ticket SD-2026-0418 | raised 2026-09-28 10:48 | from: Permit office\n"
              "The welding contractor asks how long their fire watch has to stay at the job after the welding "
              "is finished. They are quoting a copy of the procedure they were given last year.",
         gold=dict(ticket_id="SD-2026-0418", category="procedure",
                   affected_system=r"hot work|fire watch|permit|weld",
                   priority=3, action_must=[r"\b60\s*min|\bone hour\b|\b1 hour\b"],
                   action_must_not=[r"\b30\s*min"],  # HSE-PRO-012 rev 2, superseded
                   reference=r"HSE-PRO-012")),

    # --- absent: a neighbouring fact is in the corpus, and it is not the one asked for --------------
    dict(id="T08", kind="absent",
         text="Ticket SD-2026-0421 | raised 2026-09-28 11:05 | from: Operations, Unit 300\n"
              "Discharge pressure on P-301 keeps hitting the high alarm. What is the maximum discharge "
              "pressure we are allowed to run it at?",
         gold=dict(ticket_id="SD-2026-0421", category="plant_equipment", affected_system=r"p-?301",
                   priority=3,
                   action_must=[r"no (document|record|manual|entry)|not (covered|documented|found|listed|in)"
                                r"|unknown|cannot|can.?t|does not exist|no such|confirm the tag|escalat"],
                   action_must_not=[r"\d\s*barg"],  # there is no P-301: any number here is invented
                   reference=NONE_RE)),

    dict(id="T09", kind="absent",
         text="Ticket SD-2026-0423 | raised 2026-09-28 11:22 | from: Maintenance planning\n"
              "Finance want the approved budget for the K-301 major overhaul so they can raise the purchase "
              "order. Can you pull it out of the system for them?",
         gold=dict(ticket_id="SD-2026-0423", category="plant_equipment", affected_system=r"k-?301",
                   priority=4,
                   action_must=[r"no (budget|cost|document|record)|not (covered|documented|found|recorded|held)"
                                r"|unknown|cannot|can.?t|escalat|finance|planning"],
                   action_must_not=[r"\d[\d,. ]*(omr|usd|rial|dollar|million|k\b)"],
                   reference=NONE_RE)),

    # --- mixed: messy text and a document behind it ------------------------------------------------
    dict(id="T10", kind="mixed",
         text="Ticket SD-2026-0427 | raised 2026-09-28 11:40 | from: OT support\n"
              "after the patch window HS 01 started rejecting new tags for the K-302 package, engineers cant "
              "add them. log line is HX 4417 or 4471, i typed it off a photo on my phone. existing tags are "
              "still collecting fine and the archive looks ok",
         gold=dict(ticket_id="SD-2026-0427", category="historian", affected_system=r"hs-?\s?0?1|historian",
                   priority=2,  # the symptom picks HX-4417 out of the two codes: new tags rejected
                   action_must=[r"licen[cs]e|tag count|retire"],
                   action_must_not=[r"archive|queue|free space"], reference=r"MAN-HIS-01")),

    dict(id="T11", kind="mixed",
         text="Ticket SD-2026-0431 | raised 2026-09-28 12:05 | from: Control room, shift A\n"
              "EDG-01 did not start on this morning's weekly test, it cranked and stopped. The control room "
              "UPS is also beeping on and off but its display looks normal. What priority is this?",
         gold=dict(ticket_id="SD-2026-0431", category="plant_equipment", affected_system=r"edg-?0?1|generator",
                   priority=1,  # MAN-EDG-01 states priority 1 for a failure to start
                   action_must=[r"portable generator|plant manager"], action_must_not=[],
                   reference=r"MAN-EDG-01")),

    dict(id="T12", kind="mixed",
         text="Ticket SD-2026-0435 | raised 2026-09-28 12:30 | from: Rotating equipment engineering\n"
              "The compressor vendor wants remote access to EWS-01 next Tuesday to update the trend server "
              "configuration, and has asked us to open the firewall for their support laptop. What do they "
              "need from us first?",
         gold=dict(ticket_id="SD-2026-0435", category="network",
                   affected_system=r"firewall|fw-ot|ews-?0?1",
                   priority=4, action_must=[r"moc|management of change|cab|change advisory|permit"],
                   action_must_not=[], reference=r"MAN-FW-01")),
]
if not TICKETS_PATH.exists():
    TICKETS_PATH.parent.mkdir(parents=True, exist_ok=True)
    TICKETS_PATH.write_text("\n".join(json.dumps(t) for t in TICKETS) + "\n", encoding="utf-8")
    print("wrote", TICKETS_PATH)

EVAL = [json.loads(line) for line in TICKETS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
pd.DataFrame([{"id": t["id"], "kind": t["kind"], "priority": t["gold"]["priority"],
               "category": t["gold"]["category"], "reference": t["gold"]["reference"],
               "ticket": t["text"].split("\n", 1)[1][:70] + "..."} for t in EVAL])

,id,kind,priority,category,reference,ticket
0,T01,desk,3,other,^\s*(none|n/?a|-|null|not applicable|no document)\s*$,The printer in the maintenance office jams on every second page. We ar...
1,T02,desk,4,other,^\s*(none|n/?a|-|null|not applicable|no document)\s*$,New graduate engineer starts on Sunday. Please provide a second monito...
2,T03,coded,1,control_system,MAN-FGP-01,The fire and gas panel has been showing FGP-E12 since about 04:00. Nig...
3,T04,coded,2,historian,MAN-HIS-01,HS-01 has been logging HX-4471 since Saturday and trends are running a...
4,T05,coded,2,field_instrument,MAN-GD-01,"Gas detector GD-3107 failed its six-monthly calibration this morning, ..."
5,T06,stale,3,procedure,HSE-PRO-007,"Our own H2S monitors arrived today. What do we set the low alarm to, a..."
6,T07,stale,3,procedure,HSE-PRO-012,The welding contractor asks how long their fire watch has to stay at t...
7,T08,absent,3,plant_equipment,^\s*(none|n/?a|-|null|not applicable|no document)\s*$,Discharge pressure on P-301 keeps hitting the high alarm. What is the ...
8,T09,absent,4,plant_equipment,^\s*(none|n/?a|-|null|not applicable|no document)\s*$,Finance want the approved budget for the K-301 major overhaul so they ...
9,T10,mixed,2,historian,MAN-HIS-01,after the patch window HS 01 started rejecting new tags for the K-302 ...


## 4. The four arms

The prompt is assembled from three pieces, and each arm decides which pieces it gets.

| Piece | base | tuned | base + retrieval | tuned + retrieval |
|---|---|---|---|---|
| the desk's rules, written out | yes | no, it was trained on them | yes | no |
| four retrieved passages | no | no | yes | yes |
| the ticket | yes | yes | yes | yes |

A tuned model needs the first piece written down once, in the training set, not on every call. That is the part of tuning that shows up on the invoice rather than in the accuracy column. Section 10 measures it, and measures it separately for a real adapter, because the stand-in below still needs the rules in its prompt.

In [7]:
TASK = "You are the service desk assistant at the Sabkha Gas Plant (SGP). Turn the ticket into one record."

FIELDS = """\
Return JSON with exactly these fields:
  ticket_id        the ticket number as written on the ticket
  category         one value from the list below
  affected_system  the tag or system the ticket is about
  priority         an integer, 1 to 4
  action           one sentence: the next step
  reference        the doc_id that settles the action, or NONE
"""

PASSAGES = """\
Passages retrieved from the SGP document set for this ticket:
{context}

Use the passages for priority, action and reference, and prefer them over your own knowledge.
A priority stated in a passage wins. If no passage settles this ticket, set reference to NONE
and say in the action who it goes to. Do not cite a passage that does not settle it.
"""


def build_prompt(item, *, tuned: bool, hits: list | None) -> str:
    """The tuned arm is given the ticket and nothing else. Everything else is a stand-in for training."""
    parts = [TASK] if not tuned else ["Ticket to record."]
    if not tuned:
        parts += [FIELDS, DESK_RULES]
    if hits is not None:
        parts.append(PASSAGES.format(context="\n\n".join(f"[{h.source}]\n{h.text}" for h in hits)))
    parts.append("Ticket:\n" + item["text"])
    return "\n".join(parts)


print(build_prompt(EVAL[3], tuned=True, hits=None))

Ticket to record.
Ticket:
Ticket SD-2026-0409 | raised 2026-09-28 09:20 | from: OT support
HS-01 has been logging HX-4471 since Saturday and trends are running about two hours behind. Can we just restart the historian service to clear it?


### Which model is the tuned one

Day 2 ends with an adapter. Whether it is in front of you right now depends on which room you are in, so the tuned arm resolves in three steps and says out loud which one it used.

1. **The adapter**, served by Ollama or as a hosted fine-tune. Set `LAB_TUNED_MODEL` to its name.
2. **Day 2's saved run**, replayed from `facilitator/prebaked_outputs/11_tuned*`.
3. **A stand-in**: the base model with the schema enforced at decoding time and a short prompt. It is *not* the adapter. It is the cheapest thing that buys part of what the adapter buys, and knowing how much of the gap it closes is worth an hour of anybody's time before they book a GPU.

In [8]:
TUNED_MODEL, TUNED_KIND = None, "stand-in"
tuned_name = os.environ.get("LAB_TUNED_MODEL", "")  # Day 2: "sgp-ticket:tuned", or a hosted "ft:..." id
if RUN_MODE == "live" and tuned_name:
    try:
        if tuned_name.startswith("ft:"):
            TUNED_MODEL = vendor_api(tuned_name)
        else:
            ensure_ollama(tuned_name)
            TUNED_MODEL = self_hosted(tuned_name)
        TUNED_KIND = "adapter"
    except Exception as e:
        print("adapter unavailable:", e)
if TUNED_MODEL is None:
    baked = prebaked_models(PREBAKED / f"{LAB}_tuned")
    if baked:
        TUNED_MODEL, TUNED_KIND = baked[0], "prebaked"
if TUNED_MODEL is None:
    TUNED_MODEL = BASE

# The stand-in needs the desk rules it was never trained on, and the schema enforced at decoding time.
TUNED_SCHEMA = SCHEMA if TUNED_KIND == "stand-in" else None
TUNED_TRAINED = TUNED_KIND != "stand-in"

BANNER = {
    "adapter": "tuned arm: Day 2's adapter. This is the real comparison.",
    "prebaked": "tuned arm: replaying Day 2's saved run. Live numbers only for the other three arms.",
    "stand-in": ("tuned arm: NO ADAPTER FOUND, so this row is the base model with the schema enforced at\n"
                 "  decoding time and the rules still in the prompt. Read that column as 'constrained\n"
                 "  prompting', not as 'fine-tuned'. Set LAB_TUNED_MODEL to use the real thing."),
}
print(BANNER[TUNED_KIND])
print("  model:", TUNED_MODEL.label, "| schema enforced:", TUNED_SCHEMA is not None,
      "| short prompt:", TUNED_TRAINED)

tuned arm: NO ADAPTER FOUND, so this row is the base model with the schema enforced at
  decoding time and the rules still in the prompt. Read that column as 'constrained
  prompting', not as 'fine-tuned'. Set LAB_TUNED_MODEL to use the real thing.
  model: vendor API: gpt-4.1-mini | schema enforced: True | short prompt: False


## 5. What gets measured

Six numbers per ticket per arm. They are deliberately the same shape as the score tables from labs 06 and 07, so the four arms compose into one frame (interface contract 4).

| Measure | What it catches |
|---|---|
| `parse` | how the record was recovered: straight from the decoder, clean JSON, out of a fenced block, or dug out of prose |
| `schema` | six fields, no extras, category in the enum, priority an integer 1 to 4 |
| `routing` | mean of `ticket_id`, `category`, `affected_system`: the fields the ticket text settles |
| `knowledge` | mean of `priority`, `action`, `reference`: the fields a document settles |
| `evidence` | did retrieval actually deliver the document the answer needed |
| `seconds`, `prompt_tokens` | what it costs to run this way on every ticket, forever |

`routing` and `knowledge` are kept apart on purpose. An average over all six fields is the number that lets an argument run for three days: each side quotes it and neither can see which half moved.

In [9]:
def parse_record(output):
    """Recover a record from whatever the arm returned, and record how much work that took."""
    if isinstance(output, dict):
        return output, "structured"      # the decoder was constrained: nothing to parse
    text = (output or "").strip()
    fenced = re.sub(r"^\s*```(?:json)?|```\s*$", "", text, flags=re.M).strip()
    braces = text[text.find("{"): text.rfind("}") + 1] if "{" in text and "}" in text else ""
    for how, candidate in (("clean", text), ("fenced", fenced), ("salvaged", braces)):
        try:
            rec = json.loads(candidate)
        except (json.JSONDecodeError, ValueError):
            continue
        if isinstance(rec, dict):
            return rec, how
    return None, "failed"


def schema_score(rec) -> float:
    if rec is None or set(rec) != set(SCHEMA["required"]):
        return 0.0
    priority = rec.get("priority")
    return float(isinstance(priority, int) and not isinstance(priority, bool) and 1 <= priority <= 4
                 and rec.get("category") in CATEGORIES
                 and all(isinstance(rec.get(f), str) for f in SCHEMA["required"] if f != "priority"))


def field_scores(item, rec) -> dict:
    gold = item["gold"]
    if rec is None:
        return {f: 0.0 for f in FROM_TICKET + FROM_DOCS}
    text = lambda f: str(rec.get(f, ""))  # noqa: E731
    action = text("action")
    return {
        "ticket_id": float(gold["ticket_id"].lower() in text("ticket_id").lower()),
        "category": float(text("category") == gold["category"]),
        "affected_system": float(bool(re.search(gold["affected_system"], text("affected_system"), re.I))),
        "priority": float(rec.get("priority") == gold["priority"]),
        "action": float(all(re.search(p, action, re.I) for p in gold["action_must"])
                        and not any(re.search(p, action, re.I) for p in gold["action_must_not"])),
        "reference": float(bool(re.search(gold["reference"], text("reference"), re.I))),
    }


def evidence(item, hits) -> float:
    """Did retrieval put the document the answer needs in front of the model?"""
    ref = item["gold"]["reference"]
    if ref == NONE_RE or hits is None:
        return np.nan  # nothing to find: scored on whether the arm says NONE instead
    return float(any(re.search(ref, h.source, re.I) for h in hits))

## 6. Retrieval, once

Both retrieval arms see the same four passages, so the only difference between them is the model.

Two details worth copying into the capstone. The query is the **ticket body**, not the whole ticket: the number and the timestamp in the header match nothing and dilute everything else. And `evidence` is measured before any model is called, because a knowledge failure has two quite different causes and you cannot fix both with one change.

In [10]:
def query_of(item) -> str:
    return " ".join(item["text"].split("\n")[1:])  # drop the header: ticket number and timestamp are noise


HITS = {item["id"]: TEXT.search(query_of(item), k=4) for item in EVAL}

retrieval = pd.DataFrame([{"id": i["id"], "kind": i["kind"], "evidence": evidence(i, HITS[i["id"]]),
                           "top sources": ", ".join(h.source.split("/")[-1][:18] for h in HITS[i["id"]][:3])}
                          for i in EVAL])
print("evidence: 1 means the document the answer needs was retrieved. Blank means there is none to find.\n")
retrieval

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6563.04it/s]


evidence: 1 means the document the answer needs was retrieved. Blank means there is none to find.



,id,kind,evidence,top sources
0,T01,desk,NaN,"LOG-2026-06-12-D.m, LOG-2026-05-09-N.m, MAN-HMI-01.md"
1,T02,desk,NaN,"INSP-2026-025.md, HSE-PRO-007_rev4.m, LOG-2026-06-11-N.m"
2,T03,coded,1.0,"MAN-FGP-01.md, WO-2026-0176.md, MAN-FGP-01.md"
3,T04,coded,1.0,"LOG-2026-04-22-D.m, RCA-2026-003.md, MAN-HIS-01.md"
4,T05,coded,1.0,"LOG-2026-06-11-N.m, MAN-GD-01.md, MAN-GD-01.md"
5,T06,stale,1.0,"HSE-PRO-007_rev4.m, HSE-PRO-007_rev4.m, HSE-PRO-007_rev4.m"
6,T07,stale,1.0,"HSE-PRO-012_rev3.m, HSE-PRO-012_rev3.m, MAN-AD-01.md"
7,T08,absent,NaN,"MAN-P-202.md, MAN-K-301.md, MAN-P-101B_rev4.md"
8,T09,absent,NaN,"PLAN-2026.md, RCA-2026-005.md, WO-2026-0234.md"
9,T10,mixed,1.0,"RCA-2026-003.md, WO-2026-0201.md, MAN-K-302.md"


Look at the blank rows before the scored ones. Four passages came back for the jammed printer too, and they are about pumps and permits. Retrieval has no way to return nothing, so "there is no document for this" is a judgement the *model* has to make, on evidence that is actively pushing the other way.

## 7. Run the four arms

48 calls: twelve tickets, four arms. Every call is cached under `outputs/11_<arm>/`, so re-running this cell after the break costs nothing.

In [11]:
ARMS = [  # name, model, tuned?, retrieval?, schema enforced
    ("base", BASE, False, False, None),
    ("tuned", TUNED_MODEL, TUNED_TRAINED, False, TUNED_SCHEMA),
    ("rag", BASE, False, True, None),
    ("tuned_rag", TUNED_MODEL, TUNED_TRAINED, True, TUNED_SCHEMA),
]


def run_arm(name, model, tuned, retrieval, schema) -> pd.DataFrame:
    jobs = [{"item_id": i["id"], "prompt": build_prompt(i, tuned=tuned, hits=HITS[i["id"]] if retrieval else None)}
            for i in EVAL]
    recs = {r["item_id"]: r for r in run_batch(model, jobs, schema=schema, out_dir=OUT / f"{LAB}_{name}",
                                               prebaked_dir=PREBAKED / f"{LAB}_{name}", workers=WORKERS)}
    size = {j["item_id"]: len(j["prompt"]) // 4 for j in jobs}  # rough tokens, chars/4
    rows = []
    for item in EVAL:
        rec = recs.get(item["id"], {})
        record, how = parse_record(rec.get("output"))
        fields = field_scores(item, record)
        rows.append({"arm": name, "id": item["id"], "kind": item["kind"], "parse": how,
                     "schema": schema_score(record),
                     "routing": float(np.mean([fields[f] for f in FROM_TICKET])),
                     "knowledge": float(np.mean([fields[f] for f in FROM_DOCS])),
                     **{f"f_{k}": v for k, v in fields.items()},
                     "evidence": evidence(item, HITS[item["id"]]) if retrieval else np.nan,
                     "seconds": rec.get("seconds", np.nan), "prompt_tokens": size[item["id"]],
                     "record": record})
    return pd.DataFrame(rows)


RESULTS = pd.concat([run_arm(*arm) for arm in ARMS], ignore_index=True)
print(f"\n{len(RESULTS)} scored records")

vendor API: gpt-4.1-mini: 12/12 ok, 33s model time
vendor API: gpt-4.1-mini: 12/12 ok, 22s model time
vendor API: gpt-4.1-mini: 12/12 ok, 26s model time
vendor API: gpt-4.1-mini: 12/12 ok, 22s model time

48 scored records


In [12]:
ORDER = [a[0] for a in ARMS]
headline = RESULTS.groupby("arm")[["schema", "routing", "knowledge"]].mean().reindex(ORDER).round(2)
salvaged = ~RESULTS.parse.isin(["structured", "clean"])
headline["needed salvage"] = salvaged.groupby(RESULTS.arm).sum().reindex(ORDER).astype(int)
headline["usually arrived as"] = RESULTS.groupby("arm").parse.agg(lambda s: s.mode()[0]).reindex(ORDER)
headline

,schema,routing,knowledge,needed salvage,usually arrived as
arm,,,,,
base,1.0,0.89,0.42,12,fenced
tuned,1.0,0.94,0.42,0,structured
rag,1.0,1.00,0.86,12,fenced
tuned_rag,1.0,1.00,0.86,0,structured


Read the two middle columns against each other, not the average of them.

**`routing` barely moves across the four rows.** Those three fields were always in the ticket. Neither switch adds anything, because there was nothing missing.

**`knowledge` moves only when retrieval is switched on.** Tuning cannot help here and it is not a question of tuning harder. Nothing in a training run on last quarter's tickets contains what HX-4471 means, and nothing in the weights knows that the fire watch changed to sixty minutes in February.

If `schema` is already at or near 1.00 for the base arm: good, say so out loud. In 2023 that column was the whole argument for fine-tuning. On a 2026 model with the fields written out, the shape is mostly free, and the honest version of the tuning case is now about prompt length, latency and consistency under load rather than about accuracy. Section 10 puts numbers on those.

**The `needed salvage` column is the part of that argument that survives.** A valid record and a record you can `json.loads` are not the same thing: a model asked politely for JSON tends to wrap it in a markdown fence, or to open with a sentence about what it is about to return. Each of those is a line of string handling in your integration, written by somebody guessing at the failure modes. Constraining the decoder deletes that code. So does an adapter.

In [13]:
by_kind = RESULTS.pivot_table(index="kind", columns="arm", values="knowledge", aggfunc="mean")
by_kind.reindex(["desk", "coded", "stale", "absent", "mixed"])[ORDER].round(2)

arm,base,tuned,rag,tuned_rag
kind,,,,
desk,1.00,1.00,0.83,0.83
coded,0.11,0.11,1.00,1.00
stale,0.33,0.33,1.00,0.83
absent,0.67,0.67,0.67,0.67
mixed,0.22,0.22,0.78,0.89


Two rows deserve more attention than the ones that moved the way you expected.

**`desk`.** If the score *drops* when retrieval is switched on, that is not noise. Those tickets have no plant document behind them, retrieval returned four passages anyway, and something in them moved the record: a priority pulled towards whatever the passages were about, or a `reference` that now cites a document which settles nothing. Retrieval is not free even when it finds nothing, and this row is the argument for routing rather than retrieving on every request. S20 opens there.

**`absent`.** Nobody wins this one, and it is worth sitting with. These tickets name equipment the document set has never heard of. Retrieval helps a model say NONE, because nothing relevant comes back, but it does not make the model say "this tag does not exist" — it obligingly proposes checking a manual nobody ever wrote. Neither switch fixes that. Validating the tag against the equipment register before the model is called does, and that is thirty lines of code and no GPU.

In [14]:
# The verdict, read off the table rather than asserted.
best = by_kind[ORDER].idxmax(axis=1)
for kind in ["desk", "coded", "stale", "absent", "mixed"]:
    row = by_kind.loc[kind, ORDER]
    winners = ", ".join(row[row == row.max()].index)
    print(f"{kind:>7s}: best {row.max():.2f}  ({winners})")
print("\nA group where every arm ties is a group where the choice does not matter. Spend nothing on it.")

   desk: best 1.00  (base, tuned)
  coded: best 1.00  (rag, tuned_rag)
  stale: best 1.00  (rag)
 absent: best 0.67  (base, tuned, rag, tuned_rag)
  mixed: best 0.89  (tuned_rag)

A group where every arm ties is a group where the choice does not matter. Spend nothing on it.


### Retrieved and still wrong

`evidence` was measured before any model was called: it says the document the answer needs was among the four passages. Where evidence is 1 and the knowledge fields are still wrong, retrieval did its job and the model did not. That is a generation failure, and no amount of better chunking touches it.

In [15]:
missed = RESULTS[(RESULTS.evidence == 1) & (RESULTS.knowledge < 1)]
missed[["arm", "id", "kind", "knowledge", "f_priority", "f_action", "f_reference"]]

,arm,id,kind,knowledge,f_priority,f_action,f_reference
33,rag,T10,mixed,0.666667,1.0,0.0,1.0
35,rag,T12,mixed,0.666667,0.0,1.0,1.0
41,tuned_rag,T06,stale,0.666667,0.0,1.0,1.0
45,tuned_rag,T10,mixed,0.666667,1.0,0.0,1.0


That is the same split lab 07 opened with, and it is still the first question to ask of any failure: was the answer in the prompt or not. If `T10` appears above, look at it. Both historian error codes live in `MAN-HIS-01`, three lines apart in one table; the ticket describes one of them ("new tags rejected, existing tags still collecting") while quoting both. The retrieval is right and the answer is about the wrong error. The fix is a line in the prompt telling the model to match the symptom rather than the quoted code, which takes five minutes — and you can only find it because the two numbers are in separate columns.

## 8. One ticket, four answers

`T04` is the whole lab in one row. The ticket asks whether the historian service can be restarted to clear `HX-4471`, and `MAN-HIS-01` says in as many words: do not, a restart discards the write queue.

No general model knows that. It is the opposite of what a decade of IT support experience suggests, and a model tuned on ticket *shape* will produce a beautifully formatted instruction to restart the service.

In [16]:
def show(ticket_id: str) -> None:
    item = next(i for i in EVAL if i["id"] == ticket_id)
    print(item["text"], "\n" + "-" * 100)
    for arm in ORDER:
        row = RESULTS[(RESULTS.arm == arm) & (RESULTS.id == ticket_id)].iloc[0]
        rec = row.record or {}
        print(f"{arm:>10s}  P{rec.get('priority', '?')}  [{rec.get('reference', '?')}]"
              f"  {' '.join(str(rec.get('action', row.parse)).split())[:150]}")
        print(f"{'':>10s}  schema {row.schema:.0f}  routing {row.routing:.2f}  knowledge {row.knowledge:.2f}")


show("T04")

Ticket SD-2026-0409 | raised 2026-09-28 09:20 | from: OT support
HS-01 has been logging HX-4471 since Saturday and trends are running about two hours behind. Can we just restart the historian service to clear it? 
----------------------------------------------------------------------------------------------------
      base  P3  [NONE]  Verify the cause of the delayed logging on HS-01 before restarting the historian service to avoid data loss.
            schema 1  routing 1.00  knowledge 0.00
     tuned  P3  [NONE]  Restart the historian service on HS-01 to clear the logging delay.
            schema 1  routing 1.00  knowledge 0.00
       rag  P2  [MAN-HIS-01]  Check free space on the archive volume and raise a priority 2 incident with the OT administrator; do not restart the historian service while HX-4471 i
            schema 1  routing 1.00  knowledge 1.00
 tuned_rag  P2  [MAN-HIS-01]  Check free space on the archive volume and do not restart the historian service while HX-4471 is 

The gap between the arms is not a prose style. It is a priority that decides who is woken up, and an instruction that either loses the write queue or does not.

Try `show("T08")` and `show("T09")` as well. Those are the tickets with no answer in the document set, . Watch the `reference` field. The arms with no documents often say NONE for the right reason, which is that they have nothing to point at. The retrieval arms have four passages in front of them and a citation is the easiest thing in the world to produce. From inside the prompt, being handed evidence and being handed the *right* evidence look identical.

## 9. The revision that settles it

`T06` and `T07` are not asking about anything obscure. They ask what the H2S low alarm is and how long a fire watch stays. Both answers changed when the procedure was revised.

A tuned model carries the answer it was trained on. Nothing in the record says so, and no eval that was written before the revision will catch it. Here is the same retriever, over the same corpus, with one revision removed: no model was retrained between these two lines.

In [17]:
def index_without(index, doc_id: str, revision: int) -> RagIndex:
    keep = [i for i, c in enumerate(index.chunks) if not (c.doc_id == doc_id and c.revision == revision)]
    return RagIndex([index.chunks[i] for i in keep], index.embeddings[keep], index.manifest)


APRIL = index_without(TEXT, "HSE-PRO-007", 4)  # the index as it stood before rev 4 was issued
question = "personal H2S monitor low alarm setting"
for label, index, superseded in [("before rev 4 was issued", APRIL, True), ("today", TEXT, False)]:
    hit = index.search(question, k=1, include_superseded=superseded)[0]  # rev 3 was current then
    line = next((ln for ln in hit.text.splitlines() if "low alarm" in ln.lower()), hit.text[:70])
    print(f"{label:>24s} : {hit.source.split('/')[-1]:22s} {line.strip()}")

print("\nwhat the arms without documents answered on T06:")
for arm in ["base", "tuned"]:
    rec = RESULTS[(RESULTS.arm == arm) & (RESULTS.id == "T06")].iloc[0].record or {}
    print(f"  {arm:>6s}: {' '.join(str(rec.get('action', '-')).split())[:110]}")

 before rev 4 was issued : HSE-PRO-007_rev3.md    | Personal monitor low alarm | 10 ppm |
                   today : HSE-PRO-007_rev4.md    | Personal monitor low alarm | 5 ppm |

what the arms without documents answered on T06:
    base: Provide the low alarm setpoint and the H2S concentration threshold for SCBA use according to plant safety rule
   tuned: Check the HSE procedures for H2S alarm settings and SCBA requirements and inform the contractor supervisor acc


The retriever changed its answer the moment the file changed, and the revision filter from lab 07 is what kept the superseded value out of it. The cost of that update was a file copy and a re-index.

The same update to a tuned model is a new training set, a training run, an eval, a regression check against the old behaviour, and a redeploy. **Retraining is a release. Re-indexing is a file copy.** That is the argument that actually settles the tune-versus-retrieve question at OQ, and it is an operations argument, not an accuracy one. HSE revises procedures on its own schedule and does not ask whether your model has been retrained.

## 10. What each one costs

Nothing above prices the two switches. They are paid in different currencies: retrieval is paid on every call forever, tuning is paid once and again at every corpus change.

In [18]:
cost = RESULTS.groupby("arm").agg(prompt_tokens=("prompt_tokens", "mean"),
                                  seconds=("seconds", "mean")).reindex(ORDER).round(1)
cost["vs base"] = (cost.prompt_tokens / cost.prompt_tokens.loc["base"]).round(2)
print(cost.to_string())

# What a real adapter costs per call: the ticket and nothing else, because the rules are in the weights.
adapter = float(np.mean([len(build_prompt(i, tuned=True, hits=None)) // 4 for i in EVAL]))
print(f"\nan adapter needs no rules block: {adapter:.0f} tokens per ticket, "
      f"{adapter / cost.prompt_tokens.loc['base']:.0%} of the base prompt"
      + ("" if TUNED_KIND == "adapter" else "   (not what the tuned rows above are running today)"))

           prompt_tokens  seconds  vs base
arm                                       
base               451.2      2.8     1.00
tuned              451.2      1.9     1.00
rag                917.7      2.2     2.03
tuned_rag          917.7      1.9     2.03

an adapter needs no rules block: 66 tokens per ticket, 15% of the base prompt   (not what the tuned rows above are running today)


The `seconds` column includes retrieval only where the arm retrieves, and this corpus is 342 chunks on a CPU. At 50,000 chunks the retrieval step is still milliseconds; the embedder and the reranker are what grow, and the reranker is the part that will surprise you.

| | Paid once | Paid on every call | Paid again when |
|---|---|---|---|
| **Tuning** | GPU time, dataset build, eval, deploy | nothing: shorter prompts, so slightly less | the behaviour changes, or the base model is upgraded |
| **Retrieval** | ingestion, chunking, embedding, index hosting | the passages in the prompt, plus retrieval latency | never: a document change is an ingest |

The prompt-size column is the honest tuning win in 2026. A tuned model that needs no rules block and no examples costs less per call and answers faster on identical input, which matters exactly when the volume is high and the task is narrow. That is a real argument. It is not the argument anybody was making on Monday.

## 11. The decision table

This is the artifact S19 hands to S20, and the one to take into the capstone. It is written from the run above, so the numbers in it are yours, not the ones from the dry run.

In [19]:
%pip install tabulate


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
table = ROOT / "outputs" / LAB / "decision_table.md"
table.parent.mkdir(parents=True, exist_ok=True)
lines = ["# Tune versus retrieve: the decision table", "",
         f"Measured in lab 11 on {len(EVAL)} tickets, base model `{BASE.label}`, tuned arm `{TUNED_KIND}`.", "",
         "| What the failure looks like | What it is | What fixes it |",
         "|---|---|---|",
         "| Wrong shape, extra prose, an enum value you never defined | behaviour | constrain the decoder first; tune if it persists at volume |",
         "| Right shape, wrong number | knowledge | retrieval |",
         "| Right shape, right number, wrong since the procedure was revised | staleness | re-index; retraining will not reach it |",
         "| Confident answer to something no document covers | no grounding to abstain on | retrieval plus an explicit 'answer NONE' rule, and an eval that contains unanswerable tickets |",
         "| Right answer, prompt three times longer than it needs to be | cost | tune, and delete the rules block |",
         "", "## Measured", "", headline.to_markdown(), "",
         "Knowledge score by ticket group:", "", by_kind[ORDER].round(2).to_markdown(), ""]
table.write_text("\n".join(lines), encoding="utf-8")

scores = ROOT / "outputs" / LAB / "scores.jsonl"   # contract 4: one row per arm per ticket
RESULTS.drop(columns=["record"]).to_json(scores, orient="records", lines=True)
print("wrote", table, "and", scores)
print("\n".join(lines[4:10]))

wrote /Users/drpreetyrai./aiguru/outputs/11/decision_table.md and /Users/drpreetyrai./aiguru/outputs/11/scores.jsonl
| What the failure looks like | What it is | What fixes it |
|---|---|---|
| Wrong shape, extra prose, an enum value you never defined | behaviour | constrain the decoder first; tune if it persists at volume |
| Right shape, wrong number | knowledge | retrieval |
| Right shape, right number, wrong since the procedure was revised | staleness | re-index; retraining will not reach it |
| Confident answer to something no document covers | no grounding to abstain on | retrieval plus an explicit 'answer NONE' rule, and an eval that contains unanswerable tickets |


## 12. Try it, if the group is ahead

- **Move a field across the line.** `priority` is scored as knowledge because manuals state it. Rewrite `T03` so no document states a priority and re-score: it becomes a behaviour field, and the arms change places.
- **Take retrieval away from the tickets that do not need it.** Run the `rag` arm with `HITS` emptied for the `desk` group. Nothing should drop, and the prompt-size column falls. That is the routing decision S20 opens with: not every request needs the whole stack.
- **Break the retrieval, not the model.** Set `k=1` and re-run. Watch `knowledge` fall on the `mixed` tickets while `schema` and `routing` stay exactly where they were. A pipeline that reports one score cannot show you this.

## What to take away

- **They are two switches, not two answers.** Tuning changes how the model behaves. Retrieval changes what it knows. A table that keeps those apart ends the argument in four rows.
- **Diagnose before you choose.** Wrong shape is a behaviour failure. Right shape and wrong number is a knowledge failure. The team that measures one blended accuracy cannot tell them apart and will fix the wrong one.
- **Constrain the decoder before you book a GPU.** On a 2026 model most of the schema win is free. Tuning still buys shorter prompts, lower latency and consistency at volume, which is a cost argument and worth making as one.
- **Retraining is a release, re-indexing is a file copy.** Whichever changes more often, the documents or the behaviour, decides which switch carries your risk.
- **Retrieval always returns something.** On a question no document answers, it supplies material to be confidently wrong with. Put unanswerable tickets in the eval set or you will never see it.

## Facilitator: save this run as the room's fallback

In [21]:
PROMOTE = False  # facilitator only: after a good live run, keep it for when the network or a model fails
if PROMOTE and RUN_MODE == "live":
    for name, *_ in ARMS:
        promote_to_prebaked(OUT / f"{LAB}_{name}", PREBAKED / f"{LAB}_{name}")